In [1]:
import pandas as pd
import numpy as np
import sys
import os


In [2]:
sys.path.append(os.path.abspath(".."))

In [3]:
from src.data_utils import (load_data)
from src.cleaning_utils import (drop_irrelevant_columns,
                                standardize_placeholders,
                                to_category,
                                handle_missing_values,
                                log_transform_skewed_columns,
                                cap_outliers_iqr,
                                drop_duplicates,
                                group_rare_categories,
                                convert_to_datetime,
                                save_cleaned_data
                                )



In [4]:
X_train = load_data("../data/X_train.csv")
y_train = load_data("../data/y_train.csv")

df_raw = X_train.merge(y_train, on="id")

DataFrame successfully loaded.
Shape: (59400, 40)
DataFrame successfully loaded.
Shape: (59400, 2)


In [5]:
cols_to_drop = [
    "id",                # unique identifier
    "wpt_name",          # too many unique values
    "recorded_by",        # constant value
    "scheme_name",       # too many unique values
    "num_private",     # too many zeros
    "extraction_type_group",  # already captured by extraction_type
    "payment_type",     # already captured by payment
    "quantity_group",   # already captured by quantity
    "waterpoint_type_group",  # already captured by waterpoint_type
    "subvillage"       # too many unique values
]

In [6]:
df_clean = drop_irrelevant_columns(df_raw, cols_to_drop)

Dropped columns: ['id', 'wpt_name', 'num_private', 'subvillage', 'recorded_by', 'scheme_name', 'extraction_type_group', 'payment_type', 'quantity_group', 'waterpoint_type_group']


In [7]:
df_clean.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
amount_tsh,59400.0,NaN,NaN,NaN,317.650385,2997.574558,0.0,0.0,0.0,20.0,350000.0
date_recorded,59400,356,2011-03-15,572,NaN,NaN,NaN,NaN,NaN,NaN,NaN
funder,55763,1896,Government Of Tanzania,9084,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gps_height,59400.0,NaN,NaN,NaN,668.297239,693.11635,-90.0,0.0,369.0,1319.25,2770.0
installer,55745,2145,DWE,17402,NaN,NaN,NaN,NaN,NaN,NaN,NaN
longitude,59400.0,NaN,NaN,NaN,34.077427,6.567432,0.0,33.090347,34.908743,37.178387,40.345193
latitude,59400.0,NaN,NaN,NaN,-5.706033,2.946019,-11.64944,-8.540621,-5.021597,-3.326156,-0.0
basin,59400,9,Lake Victoria,10248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,59400,21,Iringa,5294,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_code,59400.0,NaN,NaN,NaN,15.297003,17.587406,1.0,5.0,12.0,17.0,99.0


In [8]:
df_clean = standardize_placeholders(df_clean)

 Placeholder values replaced:
{'funder': 781, 'installer': 781, 'management': 561, 'management_group': 561, 'payment': 8157, 'water_quality': 1876, 'quality_group': 1876, 'quantity': 789, 'source': 66, 'source_class': 278}


In [9]:
df_clean["funder"].head()

0           Roman
1         Grumeti
2    Lottery Club
3          Unicef
4     Action In A
Name: funder, dtype: str

In [11]:
df_clean = to_category(df_clean, cols=["district_code", "region_code"])

Converted to category: ['region_code', 'district_code']


In [12]:
print(df_clean[["region_code", "district_code"]].dtypes)

region_code      category
district_code    category
dtype: object


In [13]:
df_clean = convert_to_datetime(df_clean, cols=["date_recorded", "construction_year"])

Converted to datetime: ['date_recorded', 'construction_year']


In [14]:
print(df_clean[["date_recorded"]].dtypes)

date_recorded    datetime64[us]
dtype: object


In [15]:
df_clean = drop_duplicates(df_clean)

Dropped 534 duplicate rows


In [16]:
df_clean.duplicated().any()

np.False_

In [17]:
df_clean = handle_missing_values(df_clean, strategy="fill", fill_value="Unknown")

Filled missing values (excluding: [])


In [18]:
df_clean.isna().sum()

amount_tsh               0
date_recorded            0
funder                   0
gps_height               0
installer                0
longitude                0
latitude                 0
basin                    0
region                   0
region_code              0
district_code            0
lga                      0
ward                     0
population               0
public_meeting           0
scheme_management        0
permit                   0
construction_year        0
extraction_type          0
extraction_type_class    0
management               0
management_group         0
payment                  0
water_quality            0
quality_group            0
quantity                 0
source                   0
source_type              0
source_class             0
waterpoint_type          0
status_group             0
dtype: int64

In [19]:
df_clean = group_rare_categories(df_clean, cols=[
    "extraction_type",
    "waterpoint_type",
    "installer",
    "funder",
    "scheme_management",
    "management",
    "water_quality",
    "quality_group",
    "quantity",
    "payment",
    "payment_type",
    "source",
    "source_type",
    "source_class"
], top_n=20, other_label="Others")

Grouped rare categories in: ['funder', 'installer', 'scheme_management', 'extraction_type', 'management', 'payment', 'water_quality', 'quality_group', 'quantity', 'source', 'source_type', 'source_class', 'waterpoint_type']


In [20]:
for col in [
    "extraction_type",
    "waterpoint_type",
    "installer",
    "funder"
]:
    print(f"{col}: {df_clean[col].nunique()} unique values")

extraction_type: 18 unique values
waterpoint_type: 7 unique values
installer: 21 unique values
funder: 21 unique values


In [21]:
df_clean = log_transform_skewed_columns(df_clean, cols=["amount_tsh", "population"], skew_threshold=1, inplace=False)

Log transformed: amount_tsh (skew=57.55)
Log transformed: population (skew=12.62)


In [22]:
df_clean = cap_outliers_iqr(df_clean, cols=["amount_tsh", "population"], factor=1.5, inplace=False)

Outliers capped: amount_tsh
Outliers capped: population


In [23]:
save_cleaned_data(df_clean, "../data/df_clean.csv")

Saved cleaned data to: ../data/df_clean.csv
